# Phase 1 — Reading and understanding the dataset

## Step 1.1: Load the dataset

**Objective:** Read the CSV file into a Pandas DataFrame.

**Input:** `synthetic_hospital_lab_dataset_1000.csv`

**Method:** Use `pd.read_csv()`.

**Expected output:** The dataset is loaded and its first five rows are displayed.

Step 1.2: Inspect dataset structure
Objective: Understand the number of records, columns, column names, and data types.

Input: Loaded DataFrame.

Method: Use shape, columns, and info().

Expected output: Dataset dimensions and data type information.

In [1]:
from pathlib import Path
import pandas as pd
INPUT_FILE = Path("synthetic_hospital_lab_dataset_1000.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Input file:", INPUT_FILE.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
df = pd.read_csv(INPUT_FILE)


Input file: C:\Users\himats\Desktop\Work\python_coding\DPP\M5\synthetic_hospital_lab_dataset_1000.csv
Output folder: C:\Users\himats\Desktop\Work\python_coding\DPP\M5\output


In [2]:
#print("Dataset loaded successfully.")
#Step 1.2: Inspect dataset structure¶
#display(df.head())
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())


Number of rows: 1000
Number of columns: 6

Column names:
['Patient_ID', 'Test_Name', 'Test_Value', 'Unit', 'Collection_Date', 'Lab_Report_Text']


In [3]:

print("\nDataset information:")
df.info()



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Patient_ID       1000 non-null   str    
 1   Test_Name        1000 non-null   str    
 2   Test_Value       871 non-null    float64
 3   Unit             1000 non-null   str    
 4   Collection_Date  1000 non-null   str    
 5   Lab_Report_Text  1000 non-null   str    
dtypes: float64(1), str(5)
memory usage: 47.0 KB


In [4]:
missing_summary = df.isna().sum().rename("missing_count").to_frame()
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df) * 100).round(2)

print("Missing values:")
display(missing_summary)


Missing values:


,missing_count,missing_percent
Patient_ID,0,0.0
Test_Name,0,0.0
Test_Value,129,12.9
Unit,0,0.0
Collection_Date,0,0.0
Lab_Report_Text,0,0.0


In [5]:

duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)


Duplicate rows: 0


In [5]:
print("Laboratory test counts:")
display(df["Test_Name"].value_counts(dropna=False))

print("\nExample laboratory report text:")


Laboratory test counts:


Test_Name
HbA1c          119
Sodium         112
CRP            111
Potassium      106
Platelets       99
Creatinine      98
Cholesterol     98
WBC             96
Glucose         83
Hemoglobin      78
Name: count, dtype: int64


Example laboratory report text:


In [6]:
phase1_summary = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Duplicate rows", "Total missing cells"],
    "Value": [df.shape[0], df.shape[1], int(df.duplicated().sum()), int(df.isna().sum().sum())]
})
display(phase1_summary)

,Metric,Value
0,Rows,1000
1,Columns,6
2,Duplicate rows,0
3,Total missing cells,129


Phase 2 — Data cleaning and regular expressions
Step 2.1: Create a working copy
Objective: Protect the original DataFrame and prepare a separate copy for processing.

Input: Original DataFrame df.

Method: Use .copy().

Expected output: A working DataFrame called clean_df.

In [7]:
clean_df = df.copy()
print(clean_df.shape)

(1000, 6)


In [8]:
print(clean_df["Lab_Report_Text"])



0      Test=HbA1c |  Result: 6.3 % |  collected 01-26...
1      Test=Sodium |  Result: 139.45 mmol/L |  collec...
2      Test=Sodium; Result: 142.59 mmol/L; collected ...
3      Test=Sodium; Result: 146.09 mmol/L; collected ...
4      Test=Glucose; Result: 95.32 mg/dL; collected 1...
                             ...                        
995    Test=Cholesterol; Result: 180.16 mg/dL; collec...
996    Test=Glucose; Result: 135.95 mg/dL; collected ...
997     Test=HbA1c; Result: 6.44 %; collected 12/09/2023
998    Test=WBC /  Result: 5.86 10^9/L /  collected 2...
999    Test=Sodium |  Result: 142.29 mmol/L |  collec...
Name: Lab_Report_Text, Length: 1000, dtype: str


In [9]:
#Method 1 Convert it into str and lower case
cleaned_text = []
# Loop through each laboratory report
for text in clean_df["Lab_Report_Text"]: 
    if isinstance(text, str):
        text = text.lower()
    cleaned_text.append(text)
# Add cleaned text as a new column
clean_df["Lab_Report_Text_Clean"] = cleaned_text
# Display results
display(clean_df[["Lab_Report_Text", "Lab_Report_Text_Clean"]].head(10))

,Lab_Report_Text,Lab_Report_Text_Clean
0,Test=HbA1c | Result: 6.3 % | collected 01-26...,test=hba1c | result: 6.3 % | collected 01-26...
1,Test=Sodium | Result: 139.45 mmol/L | collec...,test=sodium | result: 139.45 mmol/l | collec...
2,Test=Sodium; Result: 142.59 mmol/L; collected ...,test=sodium; result: 142.59 mmol/l; collected ...
3,Test=Sodium; Result: 146.09 mmol/L; collected ...,test=sodium; result: 146.09 mmol/l; collected ...
4,Test=Glucose; Result: 95.32 mg/dL; collected 1...,test=glucose; result: 95.32 mg/dl; collected 1...
5,"Test=CRP , Result: 5.66 mg/L , collected 08-...","test=crp , result: 5.66 mg/l , collected 08-..."
6,Test=WBC; Result: 10.16 10^9/L; collected 09-1...,test=wbc; result: 10.16 10^9/l; collected 09-1...
7,"Test=Sodium , Result: 143.07 mmol/L , collec...","test=sodium , result: 143.07 mmol/l , collec..."
8,Test=Potassium; Result: 3.97 mmol/L; collected...,test=potassium; result: 3.97 mmol/l; collected...
9,Test=Glucose; Result: 110.85 mg/dL; collected ...,test=glucose; result: 110.85 mg/dl; collected ...


In [10]:


# Method 2 cleaning 
clean_df["Lab_Report_Text_Clean"] = clean_df["Lab_Report_Text"].str.lower()

# Remove extra spaces
clean_df["Lab_Report_Text_Clean"] = clean_df["Lab_Report_Text_Clean"].str.replace(r"\s+", " ", regex=True).str.strip()
# Display original and cleaned text
display(clean_df[["Lab_Report_Text", "Lab_Report_Text_Clean"]].head(10))

,Lab_Report_Text,Lab_Report_Text_Clean
0,Test=HbA1c | Result: 6.3 % | collected 01-26...,test=hba1c | result: 6.3 % | collected 01-26-2023
1,Test=Sodium | Result: 139.45 mmol/L | collec...,test=sodium | result: 139.45 mmol/l | collecte...
2,Test=Sodium; Result: 142.59 mmol/L; collected ...,test=sodium; result: 142.59 mmol/l; collected ...
3,Test=Sodium; Result: 146.09 mmol/L; collected ...,test=sodium; result: 146.09 mmol/l; collected ...
4,Test=Glucose; Result: 95.32 mg/dL; collected 1...,test=glucose; result: 95.32 mg/dl; collected 1...
5,"Test=CRP , Result: 5.66 mg/L , collected 08-...","test=crp , result: 5.66 mg/l , collected 08-09..."
6,Test=WBC; Result: 10.16 10^9/L; collected 09-1...,test=wbc; result: 10.16 10^9/l; collected 09-1...
7,"Test=Sodium , Result: 143.07 mmol/L , collec...","test=sodium , result: 143.07 mmol/l , collecte..."
8,Test=Potassium; Result: 3.97 mmol/L; collected...,test=potassium; result: 3.97 mmol/l; collected...
9,Test=Glucose; Result: 110.85 mg/dL; collected ...,test=glucose; result: 110.85 mg/dl; collected ...


In [12]:
# Toy example 
import re
text = "Patient    has   fever\nand\tcough"
clean_text = re.sub(r"\s+", " ", text)
print(clean_text)

Patient has fever and cough


In [13]:
clean_df.shape

(1000, 7)

In [14]:
# Now check sample text in test results
sample_text = "Test=HbA1c | Result: 6.3 % | collected 01-26-2023"
print("Original:", sample_text)
#print("Digits:", re.findall(r"\d+", sample_text))
#print("Words:", re.findall(r"[A-Za-z]+", sample_text))
print("Decimal result:", re.findall(r"\d+(?:\.\d+)?", sample_text))


Original: Test=HbA1c | Result: 6.3 % | collected 01-26-2023
Decimal result: ['1', '6.3', '01', '26', '2023']


## Step 2.4: Extract test name, result, unit, and collection date

**Objective:** Extract structured information from the unstructured laboratory report.

**Input:** `Lab_Report_Text_Clean`.

**Method:** Use `re.search()` with capture groups. The patterns allow different separators and common date layouts.

**Important:** The date is extracted as text only. Dates such as `05/03/2024` can be ambiguous (day-first or month-first), so this notebook does not guess their meaning.

**Expected output:** Four new columns: `Extracted_Test_Name`, `Extracted_Result`, `Extracted_Unit`, and `Extracted_Date_Text`.

In [ ]:
# TEST_PATTERN,RESULT_PATTERN,DATE_PATTERN 


In [16]:
# Test name: stop when the Result field begins, allowing |, ;, or , separators.
TEST_PATTERN = re.compile(
    r"Test\s*=\s*(.*?)\s*(?:[|;,]\s*)?Result\s*:",
    flags=re.IGNORECASE
)

# Numeric result and optional unit (e.g. 6.3 %, 139.45 mmol/L, 5.66 mg/L, 109/L).
RESULT_PATTERN = re.compile(
    r"Result\s*:\s*([-+]?\d+(?:\.\d+)?)\s*([A-Za-z0-9/%]+)?",
    flags=re.IGNORECASE
)

# Capture date text in YYYY-MM-DD / YYYY/MM/DD or day/month/year-style formats.
DATE_PATTERN = re.compile(
    r"collected\s+(\d{4}[-/]\d{1,2}[-/]\d{1,2}|\d{1,2}[-/]\d{1,2}[-/]\d{4})",
    flags=re.IGNORECASE
)


In [29]:
# Function to extract information from laboratory reports
def extract_fields(report):

    if pd.isna(report):
        return pd.Series({
            "Extracted_Test_Name": pd.NA,
            "Extracted_Result": pd.NA,
            "Extracted_Unit": pd.NA,
            "Extracted_Date_Text": pd.NA
        })

    text = str(report)

    test_match = TEST_PATTERN.search(text)
    result_match = RESULT_PATTERN.search(text)
    date_match = DATE_PATTERN.search(text)

    test_name = test_match.group(1).strip() if test_match else pd.NA
    result = float(result_match.group(1)) if result_match else pd.NA
    unit = result_match.group(2) if result_match and result_match.group(2) else pd.NA
    date_text = date_match.group(1) if date_match else pd.NA

    return pd.Series({
        "Extracted_Test_Name": test_name,
        "Extracted_Result": result,
        "Extracted_Unit": unit,
        "Extracted_Date_Text": date_text
    })
# Remove duplicate columns
clean_df = clean_df.loc[:, ~clean_df.columns.duplicated()]

# Extract information
extracted = clean_df["Lab_Report_Text_Clean"].apply(extract_fields)

# Add extracted columns
clean_df = pd.concat([clean_df, extracted], axis=1)

# Display results
display(clean_df[[
    "Lab_Report_Text_Clean",
    "Extracted_Test_Name",
    "Extracted_Result",
    "Extracted_Unit",
    "Extracted_Date_Text"
]].head(15))

,Lab_Report_Text_Clean,Extracted_Test_Name,Extracted_Test_Name,Extracted_Result,Extracted_Result,Extracted_Unit,Extracted_Unit,Extracted_Date_Text,Extracted_Date_Text
0,test=hba1c | result: 6.3 % | collected 01-26-2023,hba1c,hba1c,6.30,6.30,%,%,01-26-2023,01-26-2023
1,test=sodium | result: 139.45 mmol/l | collecte...,sodium,sodium,139.45,139.45,mmol/l,mmol/l,2023/03/31,2023/03/31
2,test=sodium; result: 142.59 mmol/l; collected ...,sodium,sodium,142.59,142.59,mmol/l,mmol/l,2024-09-08,2024-09-08
3,test=sodium; result: 146.09 mmol/l; collected ...,sodium,sodium,146.09,146.09,mmol/l,mmol/l,05/03/2024,05/03/2024
4,test=glucose; result: 95.32 mg/dl; collected 1...,glucose,glucose,95.32,95.32,mg/dl,mg/dl,16/02/2025,16/02/2025
5,"test=crp , result: 5.66 mg/l , collected 08-09...",crp,crp,5.66,5.66,mg/l,mg/l,08-09-2023,08-09-2023
6,test=wbc; result: 10.16 10^9/l; collected 09-1...,wbc,wbc,10.16,10.16,10,10,09-10-2024,09-10-2024
7,"test=sodium , result: 143.07 mmol/l , collecte...",sodium,sodium,143.07,143.07,mmol/l,mmol/l,2023/05/08,2023/05/08
8,test=potassium; result: 3.97 mmol/l; collected...,potassium,potassium,3.97,3.97,mmol/l,mmol/l,06-25-2025,06-25-2025
9,test=glucose; result: 110.85 mg/dl; collected ...,glucose,glucose,110.85,110.85,mg/dl,mg/dl,08/11/2024,08/11/2024


# Python Project Hospital dataset part 2

## Step 2.5: Evaluate extraction results

**Objective:** Measure how often Regex found each field.

**Input:** Newly extracted columns.

**Method:** Count non-missing extracted values and calculate the percentage of report rows with a match.

**Expected output:** An extraction summary. A missing extraction is a signal to inspect the source text
or improve the pattern; it is not automatically proof that the original record is wrong.

In [37]:
extracted_columns = [
    "Extracted_Test_Name",
    "Extracted_Result",
    "Extracted_Unit",
    "Extracted_Date_Text"
]

extraction_summary = pd.DataFrame({
    "field": extracted_columns,
    "matched_records": [
        clean_df[col].notna().any(axis=1).sum()
        for col in extracted_columns
    ]
})

extraction_summary["not_matched"] = (
    len(clean_df) - extraction_summary["matched_records"]
)

extraction_summary["match_percent"] = (
    extraction_summary["matched_records"] / len(clean_df) * 100
).round(2)
display(extraction_summary)




,field,matched_records,not_matched,match_percent
0,Extracted_Test_Name,1000,0,100.0
1,Extracted_Result,1000,0,100.0
2,Extracted_Unit,1000,0,100.0
3,Extracted_Date_Text,1000,0,100.0
